imports

In [1]:
import pandas as pd
from dep2pyodbc import dep2connection

Connection to database

In [2]:
channel = dep2connection("CRH")

pyodbc using windows


Get data from candidate table in CRH and put it in a dataframe

In [3]:
df_candidate = pd.read_sql("SELECT * FROM Candidate", channel)

df_candidate.info()

C:\Users\verho\AppData\Local\Temp\ipykernel_77132\2725275293.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidate = pd.read_sql("SELECT * FROM Candidate", channel)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2292756 entries, 0 to 2292755
Data columns (total 9 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   ID                 int64 
 1   InstanceID         int64 
 2   AssessmentGUID     object
 3   InstrumentClassID  int64 
 4   LanguageGUID       object
 5   OrganizationGUID   object
 6   Gender             object
 7   GenderChoice       object
 8   Qualification      object
dtypes: int64(3), object(6)
memory usage: 157.4+ MB


Use only columns that are needed in the DWH from the dataframe

In [4]:
df_candidate_dwh = df_candidate[[
    "ID", "InstanceID", "OrganizationGUID", "LanguageGUID", "GenderChoice",
    "Gender", "Qualification"
    ]]

# df_candidate_dwh.head()

Add CandidateKey by combining ID and Instance into 1 code: {ID}{InstanceID}

In [5]:
df_candidate_dwh["CandidateKey"] = df_candidate_dwh["ID"].astype(str) + df_candidate_dwh["InstanceID"].astype(str)
df_candidate_dwh["CandidateKey"].astype(int)

C:\Users\verho\AppData\Local\Temp\ipykernel_77132\3799384193.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_candidate_dwh["CandidateKey"] = df_candidate_dwh["ID"].astype(str) + df_candidate_dwh["InstanceID"].astype(str)


0                14
1                22
2                23
3                24
4                32
             ...   
2292751    50832301
2292752    50832311
2292753    50832321
2292754    50832331
2292755    50832341
Name: CandidateKey, Length: 2292756, dtype: int64

Make sure columns are in same order as DWH

In [6]:
df_candidate_dwh = df_candidate_dwh[[
    "CandidateKey", "ID", "InstanceID", "OrganizationGUID",
    "LanguageGUID", "GenderChoice", "Gender", "Qualification"
    ]]

df_candidate_dwh.rename(columns={'OrganizationGUID': 'Organisation',
                                 'LanguageGUID': 'ChosenLanguage', 
                                 'GenderChoice': 'ChosenGender',}, 
    inplace=True)

df_candidate_dwh.head()

,CandidateKey,ID,InstanceID,Organisation,ChosenLanguage,ChosenGender,Gender,Qualification
0,14,1,4,B2BABE21-1705-4750-AE99-8D1316755876,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,Gender_Male,Gender_Male,Qualification_Unknown
1,22,2,2,FDE03D4F-9C9B-4B36-9C0C-F5296C6772F1,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,Gender_Female,Gender_Female,Qualification_Unknown
2,23,2,3,4B79F6DC-F833-4721-82E3-DF6F09AE8EA2,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,None,Gender_Male,Qualification_Unknown
3,24,2,4,B2BABE21-1705-4750-AE99-8D1316755876,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,Gender_Male,Gender_Male,Qualification_Unknown
4,32,3,2,FDE03D4F-9C9B-4B36-9C0C-F5296C6772F1,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,Gender_Female,Gender_Female,Qualification_Unknown


In [7]:
df_candidate.count()

ID                   2292756
InstanceID           2292756
AssessmentGUID       2292756
InstrumentClassID    2292756
LanguageGUID         2292756
OrganizationGUID     2292756
Gender               2292756
GenderChoice         2066995
Qualification        2292756
dtype: int64

In [8]:
df_candidate_dwh.to_csv("../decoded_data/DimCandidate.csv", index=False)